In [2]:
!git clone https://github.com/facebookresearch/sam3.git

Cloning into 'sam3'...
remote: Enumerating objects: 916, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 916 (delta 5), reused 3 (delta 3), pack-reused 905 (from 2)
Receiving objects: 100% (916/916), 59.47 MiB | 7.08 MiB/s, done.
Resolving deltas: 100% (183/183), done.
Updating files: 100% (504/504), done.


In [6]:
import os

path = "/workspace/sam3/pyproject.toml"
print(os.path.exists(path))

True


In [2]:
%cd /workspace/sam3/

/workspace/sam3


In [7]:
# install requirements for training
!pip install -e ".[train,dev]"

# additional utility libs
!pip install opencv-python pillow tqdm
!pip install peft accelerate

Obtaining file:///workspace/sam3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 189.4 MB/s eta 0:00:00

In [13]:
"""
แปลง mask PNG → COCO JSON format สำหรับ SAM3

โครงสร้าง dataset ที่คาดว่ามี:
    dataset/
    ├── images/
    │   ├── IMG6865.JPG
    │   ├── IMG6868.JPG
    │   └── ...
    └── masks/
        ├── IMG6865_Wonyoung.png
        ├── IMG6865_Rei.png      ← รูปเดียวกันอาจมีหลาย mask
        └── ...

Output:
    dataset/annotations_train.json
    dataset/annotations_val.json
"""

import json
import os
import re
import glob
import random
import numpy as np
from PIL import Image
import cv2
from pathlib import Path


# ============================================================
# ตั้งค่า path — แก้ตรงนี้ให้ตรงกับ dataset ของคุณ
# ============================================================
IMAGES_DIR = "/workspace/dataset/images"       # โฟลเดอร์รูปภาพ
MASKS_DIR  = "/workspace/dataset/masks"        # โฟลเดอร์ mask PNG
OUTPUT_DIR = "/workspace/dataset"              # ที่บันทึก JSON
VAL_RATIO  = 0.1                    # 10% สำหรับ validation
RANDOM_SEED = 42
# ============================================================


def mask_to_polygon(mask_array):
    """แปลง binary mask → polygon contours แบบ COCO"""
    mask_uint8 = (mask_array > 127).astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    polygons = []
    for contour in contours:
        if len(contour) < 6:  # ต้องมีอย่างน้อย 3 จุด
            continue
        polygon = contour.flatten().tolist()
        polygons.append(polygon)
    return polygons


def mask_to_bbox(mask_array):
    """แปลง binary mask → bounding box [x, y, w, h]"""
    mask_bool = mask_array > 127
    rows = np.any(mask_bool, axis=1)
    cols = np.any(mask_bool, axis=0)
    if not rows.any():
        return None
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    return [int(cmin), int(rmin), int(cmax - cmin), int(rmax - rmin)]


def get_mask_area(mask_array):
    """คำนวณพื้นที่ mask (จำนวน pixel ขาว)"""
    return int(np.sum(mask_array > 127))


def build_coco(image_ids, image_info_map, ann_list, category_map):
    """สร้าง COCO dict"""
    categories = [
        {"id": cat_id, "name": name, "supercategory": "person"}
        for name, cat_id in category_map.items()
    ]
    images = [image_info_map[img_id] for img_id in image_ids if img_id in image_info_map]
    annotations = [a for a in ann_list if a["image_id"] in set(image_ids)]

    return {
        "images": images,
        "annotations": annotations,
        "categories": categories,
    }


def main():
    random.seed(RANDOM_SEED)

    # หา mask files ทั้งหมด
    mask_files = sorted(glob.glob(os.path.join(MASKS_DIR, "*.png")) +
                        glob.glob(os.path.join(MASKS_DIR, "*.PNG")))
    
    if not mask_files:
        print(f"❌ ไม่พบ mask PNG ในโฟลเดอร์ {MASKS_DIR}")
        print("   กรุณาแก้ MASKS_DIR ให้ถูกต้อง")
        return

    print(f"✅ พบ mask files: {len(mask_files)} ไฟล์")

    # สร้าง category map จาก member_name
    category_map = {}   # {name: id}
    image_info_map = {} # {image_id: {...}}
    ann_list = []

    image_id_map = {}   # {image_filename: image_id}
    image_id_counter = 1
    ann_id_counter = 1

    # หารูปภาพทั้งหมดก่อน
    image_exts = ["*.JPG", "*.jpg", "*.jpeg", "*.JPEG", "*.png", "*.PNG"]
    all_images = []
    for ext in image_exts:
        all_images.extend(glob.glob(os.path.join(IMAGES_DIR, ext)))
    
    print(f"✅ พบ image files: {len(all_images)} ไฟล์")

    # Map ชื่อไฟล์รูป → image_id
    for img_path in sorted(all_images):
        fname = os.path.basename(img_path)
        img = Image.open(img_path)
        w, h = img.size
        image_id_map[fname] = image_id_counter
        image_info_map[image_id_counter] = {
            "id": image_id_counter,
            "file_name": fname,
            "width": w,
            "height": h,
        }
        image_id_counter += 1

    # วนแปลงแต่ละ mask
    skipped = 0
    for mask_path in mask_files:
        mask_fname = os.path.basename(mask_path)
        # ชื่อไฟล์รูปแบบ: IMG6865_Wonyoung.png
        # แยกด้วย _ ตัวสุดท้าย
        match = re.match(r'^(.+)_([^_]+)\.png$', mask_fname, re.IGNORECASE)
        if not match:
            print(f"⚠️  ข้าม (ชื่อไม่ตรงรูปแบบ): {mask_fname}")
            skipped += 1
            continue

        image_base = match.group(1)   # เช่น IMG6865
        member_name = match.group(2)  # เช่น Wonyoung

        # หา image file (ลองหลาย extension)
        image_id = None
        for ext in [".JPG", ".jpg", ".jpeg", ".JPEG", ".png", ".PNG"]:
            candidate = image_base + ext
            if candidate in image_id_map:
                image_id = image_id_map[candidate]
                break

        if image_id is None:
            print(f"⚠️  ข้าม (ไม่พบรูป): {image_base}")
            skipped += 1
            continue

        # สร้าง category ถ้ายังไม่มี
        if member_name not in category_map:
            category_map[member_name] = len(category_map) + 1

        cat_id = category_map[member_name]

        # อ่าน mask
        mask = np.array(Image.open(mask_path).convert("L"))
        bbox = mask_to_bbox(mask)
        if bbox is None:
            print(f"⚠️  ข้าม (mask ว่าง): {mask_fname}")
            skipped += 1
            continue

        polygons = mask_to_polygon(mask)
        area = get_mask_area(mask)

        ann_list.append({
            "id": ann_id_counter,
            "image_id": image_id,
            "category_id": cat_id,
            "segmentation": polygons,
            "bbox": bbox,
            "area": area,
            "iscrowd": 0,
        })
        ann_id_counter += 1

    print(f"\n📊 สรุป:")
    print(f"   - Categories (member names): {list(category_map.keys())}")
    print(f"   - Annotations: {len(ann_list)}")
    print(f"   - ข้ามไป: {skipped}")

    # แบ่ง train/val ตาม image_id
    all_image_ids = list(image_info_map.keys())
    random.shuffle(all_image_ids)
    val_size = max(1, int(len(all_image_ids) * VAL_RATIO))
    val_ids = set(all_image_ids[:val_size])
    train_ids = set(all_image_ids[val_size:])

    train_coco = build_coco(train_ids, image_info_map, ann_list, category_map)
    val_coco   = build_coco(val_ids,   image_info_map, ann_list, category_map)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    train_path = os.path.join(OUTPUT_DIR, "annotations_train.json")
    val_path   = os.path.join(OUTPUT_DIR, "annotations_val.json")

    with open(train_path, "w") as f:
        json.dump(train_coco, f)
    with open(val_path, "w") as f:
        json.dump(val_coco, f)

    print(f"\n✅ บันทึกแล้ว:")
    print(f"   Train: {train_path}  ({len(train_ids)} รูป, {len(train_coco['annotations'])} annotations)")
    print(f"   Val:   {val_path}  ({len(val_ids)} รูป, {len(val_coco['annotations'])} annotations)")
    print(f"\n🎯 ใช้ path เหล่านี้ใน SAM3 config:")
    print(f"   img_folder: {os.path.abspath(IMAGES_DIR)}")
    print(f"   ann_file (train): {os.path.abspath(train_path)}")
    print(f"   ann_file (val):   {os.path.abspath(val_path)}")


if __name__ == "__main__":
    main()

✅ พบ mask files: 318 ไฟล์
✅ พบ image files: 303 ไฟล์

📊 สรุป:
   - Categories (member names): ['Wonyoung', 'Yujin', 'Rei', 'Leeseo', 'Liz', 'Gaeul']
   - Annotations: 318
   - ข้ามไป: 0

✅ บันทึกแล้ว:
   Train: /workspace/dataset/annotations_train.json  (273 รูป, 288 annotations)
   Val:   /workspace/dataset/annotations_val.json  (30 รูป, 30 annotations)

🎯 ใช้ path เหล่านี้ใน SAM3 config:
   img_folder: /workspace/dataset/images
   ann_file (train): /workspace/dataset/annotations_train.json
   ann_file (val):   /workspace/dataset/annotations_val.json


In [15]:
!cp /workspace/sam3/sam3/train/configs/roboflow_v100/roboflow_v100_full_ft_100_images.yaml \
   /workspace/sam3/sam3/train/configs/ive_members.yaml

In [5]:
import os
import sam3.train as t
print(os.path.dirname(t.__file__))  # หาว่าโมดูลอยู่ที่ไหน

/workspace/sam3/sam3/train


In [6]:
!find $(python -c "import sam3.train as t; import os; print(os.path.dirname(t.__file__))") -name "*.py" | sort

/workspace/sam3/sam3/train/__init__.py
/workspace/sam3/sam3/train/data/__init__.py
/workspace/sam3/sam3/train/data/coco_json_loaders.py
/workspace/sam3/sam3/train/data/collator.py
/workspace/sam3/sam3/train/data/sam3_image_dataset.py
/workspace/sam3/sam3/train/data/sam3_video_dataset.py
/workspace/sam3/sam3/train/data/torch_dataset.py
/workspace/sam3/sam3/train/loss/__init__.py
/workspace/sam3/sam3/train/loss/loss_fns.py
/workspace/sam3/sam3/train/loss/mask_sampling.py
/workspace/sam3/sam3/train/loss/sam3_loss.py
/workspace/sam3/sam3/train/loss/sigmoid_focal_loss.py
/workspace/sam3/sam3/train/masks_ops.py
/workspace/sam3/sam3/train/matcher.py
/workspace/sam3/sam3/train/nms_helper.py
/workspace/sam3/sam3/train/optim/__init__.py
/workspace/sam3/sam3/train/optim/optimizer.py
/workspace/sam3/sam3/train/optim/schedulers.py
/workspace/sam3/sam3/train/train.py
/workspace/sam3/sam3/train/trainer.py
/workspace/sam3/sam3/train/transforms/__init__.py
/workspace/sam3/sam3/train/transforms/basic.py

In [ ]:
# ดู train.py ว่า main() รับ argument อะไรบ้าง
!cat /workspace/sam3/sam3/train/train.py

In [8]:
!find /workspace/sam3 -name "*.yaml" | sort

/workspace/sam3/sam3/train/configs/eval_base.yaml
/workspace/sam3/sam3/train/configs/gold_image_evals/sam3_gold_image_attributes.yaml
/workspace/sam3/sam3/train/configs/gold_image_evals/sam3_gold_image_crowded.yaml
/workspace/sam3/sam3/train/configs/gold_image_evals/sam3_gold_image_fg_food.yaml
/workspace/sam3/sam3/train/configs/gold_image_evals/sam3_gold_image_fg_sports.yaml
/workspace/sam3/sam3/train/configs/gold_image_evals/sam3_gold_image_metaclip_nps.yaml
/workspace/sam3/sam3/train/configs/gold_image_evals/sam3_gold_image_sa1b_nps.yaml
/workspace/sam3/sam3/train/configs/gold_image_evals/sam3_gold_image_wiki_common.yaml
/workspace/sam3/sam3/train/configs/odinw13/odinw_text_and_visual.yaml
/workspace/sam3/sam3/train/configs/odinw13/odinw_text_only.yaml
/workspace/sam3/sam3/train/configs/odinw13/odinw_text_only_positive.yaml
/workspace/sam3/sam3/train/configs/odinw13/odinw_text_only_train.yaml
/workspace/sam3/sam3/train/configs/odinw13/odinw_visual_only.yaml
/workspace/sam3/sam3/trai

In [14]:
!ls /workspace/dataset/
!ls /workspace/dataset/images/ | head -10
!find /workspace/dataset/ -name "*.json" | head -20
!find /workspace/dataset/ -name "*.png" -path "*mask*" | head -10

annotations_train.json	annotations_val.json  images  masks
IMG6865.JPG
IMG6868.JPG
IMG6872.JPG
IMG6873.JPG
IMG6874.JPG
IMG6876.JPG
IMG6877.JPG
IMG6878.JPG
IMG6879.JPG
IMG6880.JPG
/workspace/dataset/annotations_val.json
/workspace/dataset/annotations_train.json
/workspace/dataset/masks/IMG9044_Rei.png
/workspace/dataset/masks/IMG7229_Yujin.png
/workspace/dataset/masks/IMG7229_Wonyoung.png
/workspace/dataset/masks/IMG7229_Rei.png
/workspace/dataset/masks/IMG7229_Liz.png
/workspace/dataset/masks/IMG7229_Leeseo.png
/workspace/dataset/masks/IMG7229_Gaeul.png
/workspace/dataset/masks/IMG7226_Yujin.png
/workspace/dataset/masks/IMG7226_Wonyoung.png
/workspace/dataset/masks/IMG7226_Rei.png
find: ‘standard output’: Broken pipe
find: write error


In [8]:
import os
print(os.path.exists("/workspace/sam3/sam3/train/configs/ive_members.yaml"))

True


In [ ]:
import os

!cd /workspace/sam3 && python -m sam3.train.train \
    --config configs/ive_members \
    --use-cluster 0 \
    --num-gpus 1

###################### Train App Config ####################
paths:
  experiment_log_dir: /workspace/outputs/ive_training
  bpe_path: /workspace/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz
ive_train:
  train_transforms:
  - _target_: sam3.train.transforms.basic_for_api.ComposeAPI
    transforms:
    - _target_: sam3.train.transforms.filter_query_transforms.FlexibleFilterFindGetQueries
      query_filter:
        _target_: sam3.train.transforms.filter_query_transforms.FilterCrowds
    - _target_: sam3.train.transforms.point_sampling.RandomizeInputBbox
      box_noise_std: 0.1
      box_noise_max: 20
    - _target_: sam3.train.transforms.segmentation.DecodeRle
    - _target_: sam3.train.transforms.basic_for_api.RandomResizeAPI
      sizes:
        _target_: sam3.train.transforms.basic.get_random_resize_scales
        size: ${scratch.resolution}
        min_size: 480
        rounded: false
      max_size:
        _target_: sam3.train.transforms.basic.get_random_resize_max_size
        s

In [12]:
!pip install einops decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 141.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
